# 7장. 손글씨 숫자 분류 모델 학습하기

『밑바닥부터 만드는 추론 엔진』 7장의 모델을 직접 학습하는 노트북입니다.

이 노트북은 선택 사항입니다. 책은 이미 학습해 둔 모델을 내려받아 쓸 수 있게
해 두었습니다(`sh scripts/download_ch07.sh`). 가중치가 어디서 오는지 직접
보고 싶을 때 실행해 보십시오.

하는 일은 세 가지입니다.

1. 저장소와 필요한 패키지를 준비한다.
2. MNIST 데이터를 내려받는다.
3. 모델을 학습하고, badak을 검증할 PyTorch 정답을 저장한다.

같은 코드로 학습해도 기계와 PyTorch 버전에 따라 가중치가 조금 다를 수
있습니다. 그래서 여기서 만든 모델의 정확도는 책의 숫자와 소수점 아래에서
다를 수 있습니다. 검증에는 문제가 없습니다. 정답 파일도 같은 모델에서 함께
만들어지기 때문입니다.

## 1. 준비

저장소 안에서 실행하면 그대로 쓰고, Colab처럼 저장소가 없는 곳에서는 내려받습니다.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if not Path("reference/ch07_train_mnist.py").exists():
    if Path("../reference/ch07_train_mnist.py").exists():
        os.chdir("..")
    else:
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/chosungmann/building-an-inference-engine-from-scratch"], check=True)
        os.chdir("building-an-inference-engine-from-scratch")
print("작업 디렉터리:", Path.cwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "reference/requirements.txt"], check=False)

## 2. 데이터 내려받기

학습용 6만 장과 시험용 1만 장을 받아 해시를 확인합니다.

In [ ]:
print(subprocess.run(["sh", "scripts/download_mnist.sh"], capture_output=True, text=True, check=True).stdout)

## 3. 학습

입력 784개, 은닉층 128개, 출력 10개인 다층 퍼셉트론을 5번 반복 학습합니다. CPU로 수십 초면 끝납니다.

In [ ]:
result = subprocess.run([sys.executable, "reference/ch07_train_mnist.py"], capture_output=True, text=True)
print(result.stdout)
print(result.stderr[-2000:])

## 4. 만들어진 파일

`model.safetensors`가 가중치이고 `reference.safetensors`가 PyTorch의 층별 출력입니다.

In [ ]:
from safetensors.numpy import load_file

for name in ("model.safetensors", "reference.safetensors"):
    path = Path("models/mnist_mlp") / name
    print(f"{name} ({path.stat().st_size}바이트)")
    for key, value in load_file(path).items():
        print(f"  {key:<14} {str(value.dtype):<8} {value.shape}")

## 5. badak으로 실행하기

Rust가 설치된 컴퓨터에서 저장소의 `models/mnist_mlp/`에 위 두 파일을 두고
다음을 실행합니다.

```bash
cargo run --release -p badak_models --example ch07_verify
```

Colab에서 실행했다면 아래 셀로 파일을 내려받을 수 있습니다.

In [ ]:
try:
    from google.colab import files

    for name in ("model.safetensors", "reference.safetensors"):
        files.download(f"models/mnist_mlp/{name}")
except ImportError:
    print("Colab이 아닙니다. 파일은 models/mnist_mlp/ 에 있습니다.")